# Signal Reconciliation — Redis DB1 (order_gateway) vs Redis DB2 (strategy_lab)

Đối chiếu **trực tiếp** signal `order_gateway` publish lên Redis DB1 (live, đường keyspace-notification mới từ 2026-09-23) với signal `strategy_lab` publish lên Redis DB2 (`past_signal`, continuous worker chạy full-history) — không qua CSV trung gian (khác `signal_reconciliation.ipynb`/`csv_vs_db2_reconciliation.ipynb`, cả 2 đều đi qua `runtime/exports/`).

Mục đích: 2 hệ tính tín hiệu độc lập nhau (order_gateway tính live theo Redis DB0/keyspace event, strategy_lab tính theo SQL/lịch full-history) — nếu cùng logic chiến lược, cùng dữ liệu nến, phải ra **đúng cùng 1 kết quả** cho cùng `(strategy, symbol, timeframe, bartime)`.

Chỉ đọc — không ghi gì lên Redis/đĩa.

**Khác biệt schema đã biết trước, notebook tự quy đổi khi đối chiếu (không phải lỗi nếu thấy khác định dạng thô):**
- `bartime`: DB1 dạng ISO8601 có giây + `Z` (`2026-09-22T10:00:00Z`); DB2 dạng `YYYY-MM-DD HH:MM` (không giây, không timezone suffix, nhưng cùng là UTC-naive).
- Hướng lệnh: DB1 field `tradeSide` (chuỗi `BUY`/`SELL`); DB2 field `signal` (số nguyên theo `ProtoOATradeSide`, `1`=BUY/`2`=SELL — cùng quy ước CSV export, xem memory `project_export_signal_ctrader_encoding`).
- `entry`: DB1 field `stopPrice` **chỉ có ở combo** (lệnh STOP); DB2 field `entry` cũng chỉ có ở combo. `ma_cross` là lệnh MARKET nên không hệ nào giữ entry cố định — chỉ đối chiếu được hướng + atr.
- `atr`: DB1 làm tròn 2 chữ số thập phân (`_required_number()` trong `signal_publisher.py`), DB2 giữ nguyên full precision — đối chiếu với dung sai `< 0.01`.

**Khác biệt phạm vi dữ liệu (không phải lỗi):** DB1 chỉ giữ tối đa `retention_max_entries` (500) signal gần nhất mỗi pair, TTL `retention_seconds` (7 ngày) mỗi HASH — nên DB1 chỉ có vài chục/vài trăm dòng, toàn bộ đều rất gần đây. DB2 không TTL/LTRIM, giữ full-history từ `defaults.signal_start_date` (2024-01-01). Nên notebook đi từ phía DB1 (ít hơn hẳn) rồi tra đúng các pair đó trên DB2 — không quét ngược từ DB2 (sẽ đọc thừa hàng trăm nghìn dòng lịch sử không liên quan).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from order_gateway.src.candle_reader import create_client as og_create_client
from order_gateway.src.candle_reader import normalize_app_config
from order_gateway.src.config import CONFIG as OG_CONFIG
from strategy_lab.src.og_signal.redis_io.client import (
    create_client as sl_create_client,
)
from strategy_lab.src.og_signal.redis_io.client import (
    load_config as sl_load_config,
)

og_config = normalize_app_config(OG_CONFIG)
output_config = og_config["redis"]["output"]
db1 = og_create_client(og_config["redis"], output_config["db"])
db1.ping()

sl_config = sl_load_config()
past_signal_config = sl_config["redis"]["past_signal"]
db2 = sl_create_client(sl_config["redis"], past_signal_config["db"])
db2.ping()

print(f"DB1 (order_gateway output)     -- db={output_config['db']} key_prefix={output_config['key_prefix']!r}")
print(f"DB2 (strategy_lab past_signal) -- db={past_signal_config['db']} key_prefix={past_signal_config['key_prefix']!r}")

DB1 (order_gateway output)     -- db=1 key_prefix='L_SIGNAL'
DB2 (strategy_lab past_signal) -- db=2 key_prefix='L_PastSignal'


In [2]:
# Đọc toàn bộ signal hiện có trên DB1.
#
# ma_cross tự có dấu "_" trong tên nên khớp theo hậu tố chiến lược đã biết
# trước, không split "_" ngây thơ (cùng lý do CLAUDE.md mục 7).
_KNOWN_STRATEGY_SUFFIXES = {"_COMBO": "combo", "_MA_CROSS": "ma_cross"}
_SIDE_TO_PROTO = {"BUY": 1, "SELL": 2}  # tradeSide (DB1, chuỗi) -> ProtoOATradeSide (DB2, số)


def parse_signal_list_key(key: str, key_prefix: str) -> tuple[str, str, str] | None:
    """(symbol, timeframe, strategy) từ 1 key LIST signal, vd L_SIGNAL_US30_H4_COMBO."""
    prefix = f"{key_prefix}_"
    if not key.startswith(prefix):
        return None
    body = key[len(prefix):]
    for suffix, strategy in _KNOWN_STRATEGY_SUFFIXES.items():
        if body.endswith(suffix):
            symbol_tf = body[: -len(suffix)]
            if "_" not in symbol_tf:
                return None
            symbol, timeframe = symbol_tf.rsplit("_", 1)
            return symbol, timeframe, strategy
    return None


rows = []
for raw_key in db1.scan_iter(match=f"{output_config['key_prefix']}_*", count=1000, _type="list"):
    parsed = parse_signal_list_key(raw_key, output_config["key_prefix"])
    if parsed is None:
        continue
    symbol, timeframe, strategy = parsed
    for stamp in db1.lrange(raw_key, 0, -1):
        fields = db1.hgetall(f"{raw_key}:{stamp}")
        if not fields:
            continue
        rows.append({
            "strategy": strategy,
            "symbol": symbol,
            "timeframe": timeframe,
            "bartime": fields.get("bartime"),
            "signal_db1": _SIDE_TO_PROTO.get(fields.get("tradeSide")),
            "side_db1": fields.get("tradeSide"),
            "entry_db1": float(fields["stopPrice"]) if "stopPrice" in fields else None,
            "atr_db1": float(fields["atr"]) if "atr" in fields else None,
        })

df_db1 = pd.DataFrame(rows)
if not df_db1.empty:
    # DB1 bartime: ISO8601 có giây + "Z" -- ép về UTC-naive để khớp DB2.
    df_db1["bartime"] = pd.to_datetime(df_db1["bartime"], utc=True).dt.tz_localize(None)
    df_db1 = df_db1.sort_values(["strategy", "symbol", "timeframe", "bartime"]).reset_index(drop=True)

db1_pairs = (
    df_db1[["strategy", "symbol", "timeframe"]].drop_duplicates().values.tolist()
    if not df_db1.empty else []
)
print(f"DB1: {len(df_db1)} signal trên {len(db1_pairs)} pair (strategy, symbol, timeframe)")
df_db1

DB1: 340 signal trên 87 pair (strategy, symbol, timeframe)


,strategy,symbol,timeframe,bartime,signal_db1,side_db1,entry_db1,atr_db1
0,combo,BTCUSD,H1,2026-09-19 20:00:00,2,SELL,80963.85,292.71
1,combo,BTCUSD,H1,2026-09-22 10:00:00,2,SELL,85661.60,551.75
2,combo,BTCUSD,H1,2026-09-23 07:00:00,2,SELL,85914.50,484.76
3,combo,BTCUSD,H2,2026-09-20 01:00:00,2,SELL,80154.45,519.69
4,combo,BTCUSD,H2,2026-09-23 07:00:00,2,SELL,85613.70,689.72
...,...,...,...,...,...,...,...,...
335,ma_cross,US500,M45,2026-09-18 16:30:00,2,SELL,NaN,12.85
336,ma_cross,US500,M45,2026-09-22 16:30:00,2,SELL,NaN,10.37
337,ma_cross,US500,M45,2026-09-22 21:00:00,2,SELL,NaN,8.65
338,ma_cross,US500,M45,2026-09-23 06:00:00,1,BUY,NaN,5.30


In [3]:
# Đọc DB2 CHỈ cho đúng các pair đang có trên DB1 (DB2 full-history rất lớn,
# quét hết 88 pair sẽ đọc thừa hàng trăm nghìn dòng không liên quan tới
# việc đang muốn kiểm -- xem ghi chú "Khác biệt phạm vi dữ liệu" ở đầu bài).
def db2_list_key(key_prefix: str, symbol: str, timeframe: str, strategy: str) -> str:
    return f"{key_prefix}_{symbol.upper()}_{timeframe.upper()}_{strategy.upper()}"


db2_frames = []
pairs_missing_on_db2 = []

for strategy, symbol, timeframe in db1_pairs:
    list_key = db2_list_key(past_signal_config["key_prefix"], symbol, timeframe, strategy)
    stamps = db2.lrange(list_key, 0, -1)
    if not stamps:
        pairs_missing_on_db2.append((strategy, symbol, timeframe))
        continue
    pipe = db2.pipeline(transaction=False)
    for stamp in stamps:
        pipe.hgetall(f"{list_key}:{stamp}")
    hashes = pipe.execute()
    rows = [h for h in hashes if h]
    if not rows:
        pairs_missing_on_db2.append((strategy, symbol, timeframe))
        continue
    frame = pd.DataFrame(rows).assign(strategy=strategy, symbol=symbol, timeframe=timeframe)
    db2_frames.append(frame)

df_db2 = pd.concat(db2_frames, ignore_index=True) if db2_frames else pd.DataFrame()
if not df_db2.empty:
    # DB2 bartime: "YYYY-MM-DD HH:MM" UTC-naive, không giây -- pandas tự suy ra format.
    df_db2["bartime"] = pd.to_datetime(df_db2["bartime"])
    df_db2 = df_db2.rename(columns={"signal": "signal_db2", "entry": "entry_db2", "atr": "atr_db2"})
    df_db2["signal_db2"] = pd.to_numeric(df_db2["signal_db2"], errors="coerce").astype("Int64")
    df_db2["atr_db2"] = pd.to_numeric(df_db2["atr_db2"], errors="coerce")
    if "entry_db2" in df_db2.columns:
        df_db2["entry_db2"] = pd.to_numeric(df_db2["entry_db2"], errors="coerce")
    else:
        df_db2["entry_db2"] = None

print(f"DB2: {len(db1_pairs) - len(pairs_missing_on_db2)}/{len(db1_pairs)} pair (của DB1) có dữ liệu trên DB2, "
      f"tổng {len(df_db2)} dòng đọc về")
if pairs_missing_on_db2:
    print(f"Chưa có trên DB2 ({len(pairs_missing_on_db2)} pair):")
    for p in pairs_missing_on_db2:
        print("  -", p)

DB2: 87/87 pair (của DB1) có dữ liệu trên DB2, tổng 70875 dòng đọc về


In [4]:
# Đối chiếu theo (strategy, symbol, timeframe, bartime) -- left join từ DB1:
# câu hỏi đang trả lời là "mọi signal order_gateway ĐÃ publish có khớp với
# những gì strategy_lab tính cho đúng bar đó không", không phải coverage
# 2 chiều (DB2 chắc chắn có nhiều bartime hơn DB1 rất nhiều, không liên quan).
if df_db1.empty:
    print("DB1 chưa có signal nào để đối chiếu.")
    comparison = pd.DataFrame()
else:
    comparison = df_db1.merge(
        df_db2,
        on=["strategy", "symbol", "timeframe", "bartime"],
        how="left",
    )
    comparison["found_in_db2"] = comparison["signal_db2"].notna()
    comparison["signal_match"] = comparison["signal_db1"] == comparison["signal_db2"]
    comparison["entry_match"] = comparison.apply(
        lambda r: True if pd.isna(r["entry_db1"]) or pd.isna(r["entry_db2"])
        else abs(r["entry_db1"] - r["entry_db2"]) < 0.01,
        axis=1,
    )
    comparison["atr_match"] = (comparison["atr_db1"] - comparison["atr_db2"]).abs() < 0.01

display_cols = [
    "strategy", "symbol", "timeframe", "bartime",
    "side_db1", "signal_db1", "signal_db2", "signal_match",
    "entry_db1", "entry_db2", "entry_match",
    "atr_db1", "atr_db2", "atr_match",
    "found_in_db2",
]
comparison[display_cols] if not comparison.empty else comparison

,strategy,symbol,timeframe,bartime,side_db1,signal_db1,signal_db2,signal_match,entry_db1,entry_db2,entry_match,atr_db1,atr_db2,atr_match,found_in_db2
0,combo,BTCUSD,H1,2026-09-19 20:00:00,SELL,2,2,True,80963.85,80963.85,True,292.71,292.71,True,True
1,combo,BTCUSD,H1,2026-09-22 10:00:00,SELL,2,2,True,85661.60,85661.60,True,551.75,551.75,True,True
2,combo,BTCUSD,H1,2026-09-23 07:00:00,SELL,2,2,True,85914.50,85914.50,True,484.76,484.76,True,True
3,combo,BTCUSD,H2,2026-09-20 01:00:00,SELL,2,2,True,80154.45,80154.45,True,519.69,519.69,True,True
4,combo,BTCUSD,H2,2026-09-23 07:00:00,SELL,2,2,True,85613.70,85613.70,True,689.72,689.72,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335,ma_cross,US500,M45,2026-09-18 16:30:00,SELL,2,2,True,NaN,NaN,True,12.85,12.85,True,True
336,ma_cross,US500,M45,2026-09-22 16:30:00,SELL,2,2,True,NaN,NaN,True,10.37,10.37,True,True
337,ma_cross,US500,M45,2026-09-22 21:00:00,SELL,2,2,True,NaN,NaN,True,8.65,8.65,True,True
338,ma_cross,US500,M45,2026-09-23 06:00:00,BUY,1,1,True,NaN,NaN,True,5.30,5.30,True,True


In [5]:
# Tổng kết.
if comparison.empty:
    print("Không có gì để tổng kết.")
else:
    total = len(comparison)
    found = int(comparison["found_in_db2"].sum())
    signal_ok = int((comparison["found_in_db2"] & comparison["signal_match"]).sum())
    entry_ok = int((comparison["found_in_db2"] & comparison["entry_match"]).sum())
    atr_ok = int((comparison["found_in_db2"] & comparison["atr_match"]).sum())

    print(f"Tổng {total} signal trên DB1.")
    print(f"  - Tìm thấy đúng bartime trên DB2: {found}/{total}")
    print(f"  - Khớp hướng BUY/SELL: {signal_ok}/{found}")
    print(f"  - Khớp entry price (chỉ combo có cả 2 bên): {entry_ok}/{found}")
    print(f"  - Khớp atr (dung sai < 0.01, DB1 làm tròn 2 chữ số còn DB2 full precision): {atr_ok}/{found}")

    mismatches = comparison[
        comparison["found_in_db2"]
        & (~comparison["signal_match"] | ~comparison["entry_match"] | ~comparison["atr_match"])
    ]
    if not mismatches.empty:
        print("\nCÁC DÒNG LỆCH GIÁ TRỊ:")
        display(mismatches[display_cols])
    else:
        print("\nKhông có dòng nào lệch giá trị.")

    not_found = comparison[~comparison["found_in_db2"]]
    if not not_found.empty:
        print(f"\n{len(not_found)} signal DB1 KHÔNG tìm thấy bartime tương ứng trên DB2 "
              f"(strategy_lab worker chưa chạy tới bar đó, hoặc lệch giờ đóng bar):")
        display(not_found[["strategy", "symbol", "timeframe", "bartime", "side_db1"]])

Tổng 340 signal trên DB1.
  - Tìm thấy đúng bartime trên DB2: 340/340
  - Khớp hướng BUY/SELL: 340/340
  - Khớp entry price (chỉ combo có cả 2 bên): 340/340
  - Khớp atr (dung sai < 0.01, DB1 làm tròn 2 chữ số còn DB2 full precision): 340/340

Không có dòng nào lệch giá trị.
